In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load the turbofan dataset (adjust the filename based on your actual file name inside the folder)
# Usually it's train_FD001.txt or similar file
data_path = "../data/Industrial Risk and Reliability Assessment/"
print("Files in directory:", os.listdir(data_path))

Files in directory: ['test_data.csv', 'train_data.csv']


In [2]:
# Load train and test datasets
train_df = pd.read_csv(data_path + 'train_data.csv')
test_df = pd.read_csv(data_path + 'test_data.csv')

print("Train Data Shape:", train_df.shape)
print("Test Data Shape:", test_df.shape)

# Display first 5 rows of train data
train_df.head()

Train Data Shape: (61249, 27)
Test Data Shape: (41214, 26)


,engine_id,cycle,op_setting_1,op_setting_2,op_setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,...,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,320
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,...,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,319
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,...,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,318
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,...,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,317
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,...,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,316


In [3]:
# Check for missing values in train and test datasets
print("Train missing values:\n", train_df.isnull().sum().sum())
print("Test missing values:\n", test_df.isnull().sum().sum())

Train missing values:
 0
Test missing values:
 0


In [4]:
# Find columns with zero variance (constant values) in train data
constant_cols = [col for col in train_df.columns if train_df[col].nunique() == 1]
print("Constant columns in train:", constant_cols)

# Drop constant columns if any, and duplicate rows
train_df = train_df.loc[:, train_df.nunique() > 1]
test_df = test_df.loc[:, test_df.nunique() > 1]

print("Train shape after dropping constant cols:", train_df.shape)

Constant columns in train: []
Train shape after dropping constant cols: (61249, 27)


In [5]:
from sklearn.preprocessing import StandardScaler

# Display all column names
print("Column Names:", train_df.columns.tolist())

Column Names: ['engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3', 'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5', 'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'RUL']


In [6]:
# Separate features (exclude engine_id, cycle, and target if present)
# Let's scale the sensor readings and operational settings
feature_cols = [col for col in train_df.columns if col not in ['engine_id', 'cycle', 'RUL']]

scaler = StandardScaler()
train_df[feature_cols] = scaler.fit_transform(train_df[feature_cols])
test_df[feature_cols] = scaler.transform(test_df[feature_cols])

print("Scaling completed successfully!")

Scaling completed successfully!


In [7]:
# Calculate RUL (Remaining Useful Life) for train data
# RUL = Max cycle of that engine - Current cycle
max_cycles = train_df.groupby('engine_id')['cycle'].max().reset_index()
max_cycles.columns = ['engine_id', 'max_cycle']
train_df = train_df.merge(max_cycles, on='engine_id', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']
train_df = train_df.drop(columns=['max_cycle'])

In [8]:
# Save the cleaned train data
train_df.to_csv('../data/Industrial Risk and Reliability Assessment/cleaned_train_data.csv', index=False)
test_df.to_csv('../data/Industrial Risk and Reliability Assessment/cleaned_test_data.csv', index=False)

print("RUL calculated and cleaned data saved successfully!")
print("Train Data Shape with RUL:", train_df.shape)

RUL calculated and cleaned data saved successfully!
Train Data Shape with RUL: (61249, 27)


In [9]:
import os
from torchvision import datasets, transforms

# Define paths for casting dataset
casting_data_path = "../data/casting_data/"

# Check if train and test folders exist
print("Folders inside casting_data:", os.listdir(casting_data_path))

Folders inside casting_data: ['test', 'train']


In [10]:
# Basic image transform (Resize to 224x224 for CNN and convert to tensor)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [11]:
# Load train dataset to verify structure
train_dataset = datasets.ImageFolder(root=os.path.join(casting_data_path, 'train'), transform=transform)
print("Casting Train Classes found:", train_dataset.classes)
print("Total casting train images:", len(train_dataset))

Casting Train Classes found: ['defective', 'normal']
Total casting train images: 6633
